# Step 2: Rod characteristics

This step collects and calculates what the simulation needs to know about
a rod:

- **Swingweight**: the moment of inertia about the rod butt, estimated
  from per-section measurements (Løvoll & Angus, 2008).
- **Stiffness and mass profiles**: bending stiffness EI(s) and mass per
  length μ(s) along the rod, fitted to filmed static bend tests, a
  tip-mass frequency test and the Common Cents System (CCS) point.

The `flycastsim` workflow has five steps. Step 1 stands on its own, and
steps 2 to 4 each produce parameters that step 5 uses.

1. Simple 1-D model (`01_simple_1d_model.ipynb`)
2. Rod characteristics (`02_rod_characteristics.ipynb`)
3. Line and leader characteristics (`03_line_and_leader.ipynb`)
4. Casting parameters (`04_casting_parameters.ipynb`)
5. Simulate a fly cast (`05_simulate_cast.ipynb`)

`complete_workflow.ipynb` runs steps 2 to 5 in sequence.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from flycastsim import (
    load_example_rods,
    plot_swingweight_contributions,
    swingweight,
)
from flycastsim.rod import (
    check_consistency,
    fit_rod,
    fitted_rods_table,
    load_fitted_rods,
    load_measured_rods,
    load_rod_measurement,
    plot_fitted_profiles,
    plot_rod_profiles,
)

## Swingweight from section measurements

For each rod section you measure the length, the mass and the balance
point (the distance from the thick end to where the section balances).
The worked examples from the paper are bundled with the package.

In [ ]:
example_rods = load_example_rods()
rod = example_rods[0]
print(rod.make, rod.model)
pd.DataFrame(rod.sections)

In [ ]:
result = swingweight(rod.sections, rod.assembled_length,
                     has_reel_seat=rod.has_reel_seat)
print(f"Estimated swingweight: {result.swingweight * 1000:.1f} g m²")
print(f"Published value:       {rod.reference_swingweight_gm2} g m²")

Mass near the tip counts far more than mass near the grip, because the
moment of inertia grows with the square of the distance to the butt.

In [ ]:
plot_swingweight_contributions(result)

## Measured rods and the consistency check

Two rods measured for this project are bundled as well. Their files also
record the mass and balance point of the assembled rod, which gives an
independent check on the section data.

In [ ]:
rows = []
for measured in load_measured_rods():
    check = check_consistency(measured)
    estimate = swingweight(measured.sections, measured.assembled_length,
                           has_reel_seat=measured.has_reel_seat)
    rows.append({
        "rod": measured.id,
        "swingweight [g m²]": round(estimate.swingweight * 1000, 1),
        "section mass [g]": round(check.section_mass * 1000, 1),
        "measured mass [g]": round(check.measured_mass * 1000, 1),
        "balance point from sections [m]": round(check.balance_point, 3),
        "measured balance point [m]": check.measured_balance_point,
    })
pd.DataFrame(rows).set_index("rod")

## Fitted stiffness and mass profiles

Three rods have been fitted from measurement data and ship with the
package. Each is ready to use in the cast simulation of step 5.

In [ ]:
fitted_rods = load_fitted_rods()
pd.DataFrame(fitted_rods_table(fitted_rods)).set_index("rod").T

In [ ]:
plot_fitted_profiles(fitted_rods)

A fitted rod gives EI and μ as functions of the distance `s` from the
butt, and a `component` that the simulation accepts directly.

In [ ]:
xp = {r.id: r for r in fitted_rods}["XP590"]
s = np.linspace(xp.s_clamp, xp.length, 5)
pd.DataFrame({"s [m]": s, "EI [N m²]": xp.EI(s), "mu [g/m]": xp.mu(s) * 1000})

## Fit a rod yourself

A fit reads a measurement folder: a `rod.yaml` description with the bend
test frames and frequency data. The data sets of the three bundled rods
are in `assets/<ROD>/` in the repo.

**The fit is switched off by default.** With the `quick` preset it takes
about 10 seconds on a laptop, and the `full` preset takes longer, so it
is left out of the default run to keep the notebook tests fast. Set
`RUN_FIT = True` to run it. The command-line version is
`examples/rod_profile_fit.py`.

In [ ]:
RUN_FIT = False

if RUN_FIT:
    measurement = load_rod_measurement(Path("../../assets/XP590"))
    fit = fit_rod(measurement, preset="quick")
    display(plot_rod_profiles(fit))